In [10]:
import os
import torch
import pandas as pd
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import classification_report
import numpy as np

from collections import Counter
from pymorphy3 import MorphAnalyzer
from sklearn.metrics import f1_score

In [2]:
input_file = r"C:\Users\тема\Desktop\Реплики.csv"
output_file = r"C:\Users\тема\Desktop\Реплики_clean.csv"

In [3]:
text_col = "Реплика"
label_col = "Эмоция"
to_drop = ["Neutral", "Неграмматично"]

emotion_map = {
    "Joy": 0,
    "Sadness": 1,
    "Surprise": 2,
    "Fear": 3,
    "Anger": 4
}

In [15]:
input_file = r"C:\Users\тема\Desktop\Реплики.csv"
df = pd.read_csv(input_file, sep=';')
df = df.iloc[:, :2]
df.columns = ["Реплика", "Эмоция"]
df = df.dropna()
df["Эмоция"] = df["Эмоция"].str.strip()


to_drop = ["Neutral", "Неграмматично"]
df = df[~df["Эмоция"].isin(to_drop)]
df = df.reset_index(drop=True)

model_path = r"C:\Users\тема\Desktop\saved_model_rubert-tiny2"
emotions = ["joy", "sadness", "surprise", "fear", "anger"]
max_len = 100
device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path).to(device)
model.eval()

texts = df["Реплика"].fillna("").astype(str).tolist()
all_preds = []

with torch.no_grad():
    for i in tqdm(range(0, len(texts), 32), desc="rubert-tiny2"):
        batch = texts[i:i+32]
        inputs = tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=max_len
        ).to(device)
        logits = model(**inputs).logits
        preds = torch.argmax(logits, dim=1).cpu().numpy()
        all_preds.extend(preds)


id2label = {0: "joy", 1: "sadness", 2: "surprise", 3: "fear", 4: "anger"}
df["pred_tiny2"] = [id2label[p] for p in all_preds]
df["Эмоция"] = df["Эмоция"].str.lower()
df["pred_tiny2"] = df["pred_tiny2"].str.lower()


output_file = r"C:\Users\тема\Desktop\predictions_tiny2.csv"
df.to_csv(output_file, index=False, sep=';')


emotion_map = {"joy": 0, "sadness": 1, "surprise": 2, "fear": 3, "anger": 4}
true_labels = df["Эмоция"].map(emotion_map).values
pred_labels = np.array(all_preds)
print("\n" + "="*50)
print("rubert-tiny2")
print("="*50)
print(classification_report(true_labels, pred_labels, target_names=emotions, digits=3))

rubert-tiny2: 100%|██████████| 161/161 [00:01<00:00, 96.42it/s]


rubert-tiny2
              precision    recall  f1-score   support

         joy      0.424     0.230     0.298      1290
     sadness      0.370     0.234     0.287      1191
    surprise      0.497     0.492     0.494      1130
        fear      0.427     0.190     0.263       584
       anger      0.284     0.696     0.404       939

    accuracy                          0.369      5134
   macro avg      0.400     0.369     0.349      5134
weighted avg      0.402     0.369     0.354      5134



In [12]:
df["Эмоция"] = df["Эмоция"].str.strip().str.lower()
df["pred_tiny2"] = df["pred_tiny2"].str.strip().str.lower()

morph = MorphAnalyzer(lang='ru')

def analyze_row(text):
    text = str(text).strip()
    if text.endswith("?"):
        punct = "вопрос"
    elif text.endswith("!"):
        punct = "восклицание"
    elif text.endswith("."):
        punct = "утверждение"
    else:
        punct = "прочее"
    has_ne = "не" in str(text).lower().split()
    verb_lemma = None
    form_type = "прочее"
    for w in str(text).split():
        clean = w.strip('.,!?;:()""«»—…')
        if not clean:
            continue
        p = morph.parse(clean)[0]
        tag = p.tag
        if tag.POS in ('VERB'):
            verb_lemma = p.normal_form
            if tag.mood == 'impr':
                form_type = "повелительное"
            elif tag.tense == 'past':
                form_type = "прошедшее"
            break
    return verb_lemma, form_type, punct, has_ne

info = df["Реплика"].apply(analyze_row)
df["verb_lemma"] = [x[0] for x in info]
df["form_type"] = [x[1] for x in info]
df["punct"] = [x[2] for x in info]
df["has_ne"] = [x[3] for x in info]

df = df[df["form_type"].isin(["прошедшее", "повелительное"])].copy()

def get_subgroup(row):
    ne = "с_не" if row["has_ne"] else "без_не"
    return f"{row['Эмоция']} | {row['form_type']} | {ne} | {row['punct']}"

df["subgroup"] = df.apply(get_subgroup, axis=1)

subgroup_stats = []
for sg, group in df.groupby("subgroup"):
    total = len(group)
    acc = (group["Эмоция"] == group["pred_tiny2"]).mean()

    true_emo = group["Эмоция"].iloc[0]
    true_binary = (group["Эмоция"] == true_emo).astype(int).values
    pred_binary = (group["pred_tiny2"] == true_emo).astype(int).values
    f1 = f1_score(true_binary, pred_binary, zero_division=0)

    subgroup_stats.append({
        "subgroup": sg,
        "total": total,
        "acc_tiny2": acc,
        "f1_tiny2": f1,
    })

stats_df = pd.DataFrame(subgroup_stats)
stats_df = stats_df.sort_values("f1_tiny2", ascending=False)

print("\n" + "=" * 85)
print("  Accuracy и F1 по подгруппам")
print("=" * 85)
print(f"{'Подгруппа':<55s} {'N':>4s} {'Acc':>10s} {'F1':>10s}")
print("-" * 85)
for _, r in stats_df.iterrows():
    print(f"  {r['subgroup']:<53s} {r['total']:>4d} {r['acc_tiny2']:>10.4f} {r['f1_tiny2']:>10.4f}")

stats_df.to_csv(r"C:\Users\тема\Desktop\subgroup_summary_tiny2.csv", index=False, sep=';')



  Accuracy и F1 по подгруппам
Подгруппа                                                  N        Acc         F1
-------------------------------------------------------------------------------------
  anger | повелительное | с_не | утверждение              53     0.7925     0.8842
  anger | прошедшее | без_не | восклицание                98     0.7755     0.8736
  anger | прошедшее | без_не | утверждение                82     0.7561     0.8611
  anger | повелительное | с_не | восклицание             199     0.7387     0.8497
  surprise | повелительное | без_не | вопрос             110     0.7364     0.8482
  anger | повелительное | без_не | восклицание           117     0.7265     0.8416
  sadness | прошедшее | без_не | утверждение             147     0.7143     0.8333
  joy | прошедшее | без_не | утверждение                  91     0.7033     0.8258
  anger | повелительное | без_не | утверждение            95     0.6947     0.8199
  joy | прошедшее | без_не | восклицание             

In [14]:
BASE_TYPE = "прошедшее+утверждение"

operation_order = [
    "прошедшее+вопрос",
    "прошедшее+восклицание",
    "прошедшее+утверждение+не",
    "повелительное+утверждение",
    "повелительное+восклицание",
    "повелительное+вопрос",
    "повелительное+утверждение+не",
    "повелительное+восклицание+не",
    "повелительное+вопрос+не",
]

def get_type(form_type, punct, has_ne):
    ne_str = "+не" if has_ne else ""
    return f"{form_type}+{punct}{ne_str}"

df["type"] = df.apply(lambda r: get_type(r["form_type"], r["punct"], r["has_ne"]), axis=1)
groups = df.groupby("verb_lemma")

emotion_order = ["joy", "sadness", "surprise", "fear", "anger"]

print("-" * 80)
print("  Предсказания модели: базовая форма → операция")
print("  (базовая форма: прошедшее + утверждение + без 'не')")
print("-" * 80)

for target_emo in emotion_order:
    base_df = df[(df["type"] == BASE_TYPE) & (df["Эмоция"] == target_emo)]
    if len(base_df) == 0:
        continue

    print(f"\n{'-'*80}")
    print(f"  Эмоция: {target_emo}  (истинная эмоция, прошедшее+утверждение)")
    print(f"  Базовых реплик: {len(base_df)}")
    print(f"{'-'*80}")


    base_preds = Counter(base_df["pred_tiny2"])
    print(f"\n  На базовой форме:")
    print(f"    tiny2: {dict(base_preds)}")


    for op in operation_order:
        pred_changes = Counter()
        count = 0

        for _, base_row in base_df.iterrows():
            lemma = base_row["verb_lemma"]
            if lemma is None:
                continue
            group = groups.get_group(lemma)
            for _, row in group.iterrows():
                if row["type"] != op:
                    continue
                if row["Реплика"] == base_row["Реплика"]:
                    continue
                count += 1
                pred_changes[row["pred_tiny2"]] += 1

        if count == 0:
            continue

        print(f"\n  Операция: {op}  (вариаций: {count})")
        print(f"    tiny2: ", end="")
        for emo, cnt in sorted(pred_changes.items(), key=lambda x: x[1], reverse=True):
            mark = "✓" if emo == target_emo else "✗"
            print(f"{emo}={cnt}({cnt/count*100:.0f}%){mark} ", end="")
        print()

--------------------------------------------------------------------------------
  Предсказания модели: базовая форма → операция
  (базовая форма: прошедшее + утверждение + без 'не')
--------------------------------------------------------------------------------

--------------------------------------------------------------------------------
  Эмоция: joy  (истинная эмоция, прошедшее+утверждение)
  Базовых реплик: 91
--------------------------------------------------------------------------------

  На базовой форме:
    tiny2: {'joy': 64, 'anger': 16, 'sadness': 9, 'surprise': 2}

  Операция: прошедшее+вопрос  (вариаций: 90)
    tiny2: surprise=44(49%)✗ joy=34(38%)✓ anger=12(13%)✗ 

  Операция: прошедшее+восклицание  (вариаций: 90)
    tiny2: joy=71(79%)✓ anger=16(18%)✗ surprise=2(2%)✗ sadness=1(1%)✗ 

  Операция: прошедшее+утверждение+не  (вариаций: 83)
    tiny2: joy=36(43%)✓ sadness=28(34%)✗ anger=17(20%)✗ surprise=2(2%)✗ 

  Операция: повелительное+утверждение  (вариаций: 85)
  

In [16]:
input_file = r"C:\Users\тема\Desktop\Реплики.csv"
df = pd.read_csv(input_file, sep=';')
df = df.iloc[:, :2]
df.columns = ["Реплика", "Эмоция"]
df = df.dropna()
df["Эмоция"] = df["Эмоция"].str.strip()

to_drop = ["Neutral", "Неграмматично"]
df = df[~df["Эмоция"].isin(to_drop)]
df = df.reset_index(drop=True)

model_path = r"C:\Users\тема\Desktop\saved_model_rubert-base-cased"
emotions = ["joy", "sadness", "surprise", "fear", "anger"]
max_len = 100
device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path).to(device)
model.eval()

texts = df["Реплика"].fillna("").astype(str).tolist()
all_preds = []

with torch.no_grad():
    for i in tqdm(range(0, len(texts), 32), desc="rubert-base-cased"):
        batch = texts[i:i+32]
        inputs = tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=max_len
        ).to(device)
        logits = model(**inputs).logits
        preds = torch.argmax(logits, dim=1).cpu().numpy()
        all_preds.extend(preds)

id2label = {0: "joy", 1: "sadness", 2: "surprise", 3: "fear", 4: "anger"}
df["pred_base"] = [id2label[p] for p in all_preds]
df["Эмоция"] = df["Эмоция"].str.lower()
df["pred_base"] = df["pred_base"].str.lower()

output_file = r"C:\Users\тема\Desktop\predictions_base.csv"
df.to_csv(output_file, index=False, sep=';')
print(f"\nСохранено в {output_file}")
print(f"Колонки: {list(df.columns)}")

emotion_map = {"joy": 0, "sadness": 1, "surprise": 2, "fear": 3, "anger": 4}
true_labels = df["Эмоция"].map(emotion_map).values
pred_labels = np.array(all_preds)
print("\n" + "="*50)
print("rubert-base-cased")
print("="*50)
print(classification_report(true_labels, pred_labels, target_names=emotions, digits=3))


rubert-base-cased: 100%|██████████| 161/161 [00:29<00:00,  5.51it/s]


Сохранено в C:\Users\тема\Desktop\predictions_base.csv
Колонки: ['Реплика', 'Эмоция', 'pred_base']

rubert-base-cased
              precision    recall  f1-score   support

         joy      0.451     0.291     0.354      1290
     sadness      0.436     0.226     0.298      1191
    surprise      0.521     0.690     0.594      1130
        fear      0.499     0.288     0.365       584
       anger      0.299     0.590     0.397       939

    accuracy                          0.418      5134
   macro avg      0.441     0.417     0.402      5134
weighted avg      0.441     0.418     0.403      5134



In [17]:
df = pd.read_csv(r"C:\Users\тема\Desktop\predictions_base.csv", sep=';')
df["Эмоция"] = df["Эмоция"].str.strip().str.lower()
df["pred_base"] = df["pred_base"].str.strip().str.lower()


def analyze_row(text):
    text = str(text).strip()
    if text.endswith("?"):
        punct = "вопрос"
    elif text.endswith("!"):
        punct = "восклицание"
    elif text.endswith("."):
        punct = "утверждение"
    else:
        punct = "прочее"
    has_ne = "не" in str(text).lower().split()
    verb_lemma = None
    form_type = "прочее"
    for w in str(text).split():
        clean = w.strip('.,!?;:()""«»—…')
        if not clean:
            continue
        p = morph.parse(clean)[0]
        tag = p.tag
        if tag.POS in ('VERB'):
            verb_lemma = p.normal_form
            if tag.mood == 'impr':
                form_type = "повелительное"
            elif tag.tense == 'past':
                form_type = "прошедшее"
            break
    return verb_lemma, form_type, punct, has_ne

info = df["Реплика"].apply(analyze_row)
df["verb_lemma"] = [x[0] for x in info]
df["form_type"] = [x[1] for x in info]
df["punct"] = [x[2] for x in info]
df["has_ne"] = [x[3] for x in info]

df = df[df["form_type"].isin(["прошедшее", "повелительное"])].copy()

def get_subgroup(row):
    ne = "с_не" if row["has_ne"] else "без_не"
    return f"{row['Эмоция']} | {row['form_type']} | {ne} | {row['punct']}"

df["subgroup"] = df.apply(get_subgroup, axis=1)

subgroup_stats = []
for sg, group in df.groupby("subgroup"):
    total = len(group)
    acc = (group["Эмоция"] == group["pred_base"]).mean()

    true_emo = group["Эмоция"].iloc[0]
    true_binary = (group["Эмоция"] == true_emo).astype(int).values
    pred_binary = (group["pred_base"] == true_emo).astype(int).values
    f1 = f1_score(true_binary, pred_binary, zero_division=0)

    subgroup_stats.append({
        "subgroup": sg,
        "total": total,
        "acc_base": acc,
        "f1_base": f1,
    })

stats_df = pd.DataFrame(subgroup_stats)
stats_df = stats_df.sort_values("f1_base", ascending=False)

print("\n" + "=" * 85)
print("  Accuracy и F1 по подгруппам (rubert-base-cased)")
print("=" * 85)
print(f"{'Подгруппа':<55s} {'N':>4s} {'Acc':>10s} {'F1':>10s}")
print("-" * 85)
for _, r in stats_df.iterrows():
    print(f"  {r['subgroup']:<53s} {r['total']:>4d} {r['acc_base']:>10.4f} {r['f1_base']:>10.4f}")

stats_df.to_csv(r"C:\Users\тема\Desktop\subgroup_summary_base.csv", index=False, sep=';')


  Accuracy и F1 по подгруппам (rubert-base-cased)
Подгруппа                                                  N        Acc         F1
-------------------------------------------------------------------------------------
  anger | повелительное | с_не | утверждение              53     0.8113     0.8958
  joy | прошедшее | без_не | утверждение                  91     0.8022     0.8902
  surprise | прошедшее | без_не | вопрос                 136     0.7941     0.8852
  surprise | повелительное | без_не | вопрос             110     0.7909     0.8832
  surprise | прошедшее | с_не | вопрос                   377     0.7905     0.8830
  joy | прошедшее | без_не | восклицание                 114     0.7632     0.8657
  anger | повелительное | с_не | восклицание             199     0.7538     0.8596
  surprise | прошедшее | без_не | утверждение             23     0.7391     0.8500
  joy | повелительное | без_не | восклицание             101     0.7327     0.8457
  anger | прошедшее | без_не | ут

In [18]:
BASE_TYPE = "прошедшее+утверждение"

operation_order = [
    "прошедшее+вопрос",
    "прошедшее+восклицание",
    "прошедшее+утверждение+не",
    "повелительное+утверждение",
    "повелительное+восклицание",
    "повелительное+вопрос",
    "повелительное+утверждение+не",
    "повелительное+восклицание+не",
    "повелительное+вопрос+не",
]

def get_type(form_type, punct, has_ne):
    ne_str = "+не" if has_ne else ""
    return f"{form_type}+{punct}{ne_str}"

df["type"] = df.apply(lambda r: get_type(r["form_type"], r["punct"], r["has_ne"]), axis=1)
groups = df.groupby("verb_lemma")

emotion_order = ["joy", "sadness", "surprise", "fear", "anger"]

print("-" * 80)
print("  Предсказания модели: базовая форма → операция")
print("  rubert-base-cased")
print("  (базовая форма: прошедшее + утверждение + без 'не')")
print("-" * 80)

for target_emo in emotion_order:
    base_df = df[(df["type"] == BASE_TYPE) & (df["Эмоция"] == target_emo)]
    if len(base_df) == 0:
        continue

    print(f"\n{'-'*80}")
    print(f"  Эмоция: {target_emo}  (истинная эмоция, прошедшее+утверждение)")
    print(f"  Базовых реплик: {len(base_df)}")
    print(f"{'-'*80}")

    base_preds = Counter(base_df["pred_base"])
    print(f"\n  На базовой форме:")
    print(f"    base: {dict(base_preds)}")

    for op in operation_order:
        pred_changes = Counter()
        count = 0

        for _, base_row in base_df.iterrows():
            lemma = base_row["verb_lemma"]
            if lemma is None:
                continue
            group = groups.get_group(lemma)
            for _, row in group.iterrows():
                if row["type"] != op:
                    continue
                if row["Реплика"] == base_row["Реплика"]:
                    continue
                count += 1
                pred_changes[row["pred_base"]] += 1

        if count == 0:
            continue

        print(f"\n  Операция: {op}  (вариаций: {count})")
        print(f"    base: ", end="")
        for emo, cnt in sorted(pred_changes.items(), key=lambda x: x[1], reverse=True):
            mark = "✓" if emo == target_emo else "✗"
            print(f"{emo}={cnt}({cnt/count*100:.0f}%){mark} ", end="")
        print()

--------------------------------------------------------------------------------
  Предсказания модели: базовая форма → операция
  rubert-base-cased
  (базовая форма: прошедшее + утверждение + без 'не')
--------------------------------------------------------------------------------

--------------------------------------------------------------------------------
  Эмоция: joy  (истинная эмоция, прошедшее+утверждение)
  Базовых реплик: 91
--------------------------------------------------------------------------------

  На базовой форме:
    base: {'joy': 73, 'sadness': 5, 'surprise': 3, 'anger': 8, 'fear': 2}

  Операция: прошедшее+вопрос  (вариаций: 90)
    base: surprise=68(76%)✗ joy=17(19%)✓ anger=5(6%)✗ 

  Операция: прошедшее+восклицание  (вариаций: 90)
    base: joy=75(83%)✓ anger=8(9%)✗ surprise=4(4%)✗ fear=2(2%)✗ sadness=1(1%)✗ 

  Операция: прошедшее+утверждение+не  (вариаций: 83)
    base: joy=43(52%)✓ sadness=18(22%)✗ anger=17(20%)✗ surprise=4(5%)✗ fear=1(1%)✗ 

  Операция